# Lab 04 support · Expected correction values



[Day 3](README.md) · [Walkthrough](labs/lab04/WALKTHROUGH.md)

## Goal and scope

Calculate the expected business change directly from the fixed data. This notebook runs Python only: it does not create Delta versions, execute ACID transactions or simulate a successful engine.



## 1. Locate the source



In [ ]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset, new_workspace, write_json
require_fixed_dataset(SOURCE)
from masar.delta_reference import day03_reference, revision_merge
from masar.silver_reference import reference_result
WORK = new_workspace(ROOT, "day03_reference")
print("Source checked | MASAR_SMALL_V1 | Reference business values only")

## 2. Compare one changed trip



In [ ]:
report = day03_reference(SOURCE)
for label in ("corrected_trip_before", "corrected_trip_after"):
    row = report[label]
    print(label, {key: row[key] for key in ("trip_id", "fare_sar", "source_revision")})
assert report["before"]["rows"] == report["after"]["rows"] == 75

## 3. Reconcile the totals



In [ ]:
from decimal import Decimal
change = Decimal(report["after"]["fare_sar"]) - Decimal(report["before"]["fare_sar"])
assert change == Decimal("5.00")
print("Expected total before:", report["before"]["fare_sar"], "SAR")
print("Expected total after: ", report["after"]["fare_sar"], "SAR")
print("Expected change:      ", change, "SAR")

## 4. Check replay and older data



In [ ]:
assert report["after"]["digest"] == report["replay_digest"] == report["stale_replay_digest"]
print("Identical correction replay preserves values: True")
print("Older revision did not undo the correction: True")
print("Older trip revision ignored:", report["stale_ignored_count"])

## 5. Check the schema scenario meaning



In [ ]:
fixture = report["schema_fixture"]
print(json.dumps({k:fixture[k] for k in ("trip_id","extra_column","spark_type","value","meaning")}, indent=2))
print("Expected sandbox layout: 74 existing null surcharges + 1 supplied value")
print("No schema change was executed in this reference notebook.")

## 6. Save verified arithmetic, not engine claims



In [ ]:
assert all(v is True for v in report["checks"].values())
assert report["engine_executed"] is False
write_json(WORK / "day03_reference.json", report)
print("Reference checks passed:", len(report["checks"]))
print("Engine executed:", report["engine_executed"])
print("Next: the actual Lab 4a and 4b notebooks, once the runtime is provisioned.")